In [0]:
# ✨ USE YOUR REAL VALUES BELOW

client_id      = ""     # Application (client) ID
tenant_id      = ""      # Directory (tenant) ID
client_secret  = ""          # Client Secret (Value)

storage_account = "azuresupply2605"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


In [0]:
external_silver = spark.read.parquet(
    "abfss://silver@azuresupply2605.dfs.core.windows.net/external/"
)

feature_silver = spark.read.parquet(
    "abfss://silver@azuresupply2605.dfs.core.windows.net/feature/"
)

demand_silver = spark.read.parquet(
    "abfss://silver@azuresupply2605.dfs.core.windows.net/demand/"
)

In [0]:
# Step 1: Validate Silver Tables

print("=== SILVER TABLES CHECK ===")
print("Feature rows:", feature_silver.count())
print("Demand rows:", demand_silver.count())
print("External rows:", external_silver.count())

print("\n=== FEATURE SILVER SCHEMA ===")
feature_silver.printSchema()

print("\n=== DEMAND SILVER SCHEMA ===")
demand_silver.printSchema()

print("\n=== EXTERNAL SILVER SCHEMA ===")
external_silver.printSchema()


=== SILVER TABLES CHECK ===
Feature rows: 10962
Demand rows: 10962
External rows: 1827

=== FEATURE SILVER SCHEMA ===
root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- service: string (nullable = true)
 |-- daily_usage_units: integer (nullable = true)
 |-- peak_usage_units: integer (nullable = true)
 |-- vm_count: integer (nullable = true)
 |-- storage_tb: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- econ_index: integer (nullable = true)
 |-- downtime_min: integer (nullable = true)
 |-- usage_lag_1: double (nullable = true)
 |-- usage_lag_7: double (nullable = true)
 |-- week_over_week_growth: double (nullable = true)
 |-- seasonality_factor: double (nullable = true)


=== DEMAND SILVER SCHEMA ===
root
 |-- daily_usage_units: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- downtime_min: integer (nullable = true)
 |-- econ_index: integer (nullable = true)
 |-- peak_usage_units: integer (nullable = true)
 |-- region

In [0]:
from pyspark.sql.functions import col

# Step 2: Rename demand columns to avoid name conflicts
demand_renamed = (
    demand_silver
    .withColumnRenamed("date", "d_date")
    .withColumnRenamed("region", "d_region")
    .withColumnRenamed("service", "d_service")
    .withColumnRenamed("daily_usage_units", "d_daily_usage_units")
    .withColumnRenamed("peak_usage_units", "d_peak_usage_units")
    .withColumnRenamed("vm_count", "d_vm_count")
    .withColumnRenamed("storage_tb", "d_storage_tb")
    .withColumnRenamed("season", "d_season")
    .withColumnRenamed("econ_index", "d_econ_index")
    .withColumnRenamed("downtime_min", "d_downtime_min")
)

print("Renamed Demand Schema:")
demand_renamed.printSchema()


Renamed Demand Schema:
root
 |-- d_daily_usage_units: integer (nullable = true)
 |-- d_date: date (nullable = true)
 |-- d_downtime_min: integer (nullable = true)
 |-- d_econ_index: integer (nullable = true)
 |-- d_peak_usage_units: integer (nullable = true)
 |-- d_region: string (nullable = true)
 |-- d_season: string (nullable = true)
 |-- d_service: string (nullable = true)
 |-- d_storage_tb: integer (nullable = true)
 |-- d_vm_count: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col

# Step 3: Join Feature with renamed Demand
gold_step1 = (
    feature_silver.alias("f")
    .join(
        demand_renamed.alias("d"),
        (col("f.date") == col("d.d_date")) &
        (col("f.region") == col("d.d_region")) &
        (col("f.service") == col("d.d_service")),
        "left"
    )
)

print("Rows after Feature + Demand Join:", gold_step1.count())
gold_step1.printSchema()
display(gold_step1.limit(5))


Rows after Feature + Demand Join: 10962
root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- service: string (nullable = true)
 |-- daily_usage_units: integer (nullable = true)
 |-- peak_usage_units: integer (nullable = true)
 |-- vm_count: integer (nullable = true)
 |-- storage_tb: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- econ_index: integer (nullable = true)
 |-- downtime_min: integer (nullable = true)
 |-- usage_lag_1: double (nullable = true)
 |-- usage_lag_7: double (nullable = true)
 |-- week_over_week_growth: double (nullable = true)
 |-- seasonality_factor: double (nullable = true)
 |-- d_daily_usage_units: integer (nullable = true)
 |-- d_date: date (nullable = true)
 |-- d_downtime_min: integer (nullable = true)
 |-- d_econ_index: integer (nullable = true)
 |-- d_peak_usage_units: integer (nullable = true)
 |-- d_region: string (nullable = true)
 |-- d_season: string (nullable = true)
 |-- d_service: string (nullable = 

date,region,service,daily_usage_units,peak_usage_units,vm_count,storage_tb,season,econ_index,downtime_min,usage_lag_1,usage_lag_7,week_over_week_growth,seasonality_factor,d_daily_usage_units,d_date,d_downtime_min,d_econ_index,d_peak_usage_units,d_region,d_season,d_service,d_storage_tb,d_vm_count
2022-10-06,East US,Compute,157899,175674,10465,0,Autumn,109,0,79757.0,105464.0,49.71838731699916,1.1379450666416375,157899,2022-10-06,0,109,175674,East US,Autumn,Compute,0,10465
2023-09-11,East US,Compute,166349,195137,13989,0,Autumn,92,1,85447.0,117923.0,41.065780212511555,1.1743873636069189,166349,2023-09-11,1,92,195137,East US,Autumn,Compute,0,13989
2020-02-10,East US,Storage,0,0,0,855,Winter,109,0,0.0,0.0,0.0,1.1490928323493095,0,2020-02-10,0,109,0,East US,Winter,Storage,855,0
2020-05-08,East US,Storage,0,0,0,977,Spring,100,1,0.0,0.0,0.0,1.0769441644619129,0,2020-05-08,1,100,0,East US,Spring,Storage,977,0
2021-02-02,East US,Storage,0,0,0,743,Winter,104,0,0.0,0.0,0.0,1.054067234234836,0,2021-02-02,0,104,0,East US,Winter,Storage,743,0


In [0]:
from pyspark.sql.functions import col

# Step 3: Re-join Feature + Demand (this part was fine)
gold_step1 = (
    feature_silver.alias("f")
    .join(
        demand_renamed.alias("d"),
        (col("f.date") == col("d.d_date")) &
        (col("f.region") == col("d.d_region")) &
        (col("f.service") == col("d.d_service")),
        "left"
    )
)

# Step 4: Join with External — FIXED VERSION
# Rename external.date BEFORE joining to avoid creating duplicates.
external_fixed = external_silver.withColumnRenamed("date", "external_date")

gold_step2 = (
    gold_step1.alias("g")
    .join(
        external_fixed.alias("e"),
        col("g.date") == col("e.external_date"),
        "left"
    )
)

gold_step2.printSchema()
display(gold_step2.limit(3))


root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- service: string (nullable = true)
 |-- daily_usage_units: integer (nullable = true)
 |-- peak_usage_units: integer (nullable = true)
 |-- vm_count: integer (nullable = true)
 |-- storage_tb: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- econ_index: integer (nullable = true)
 |-- downtime_min: integer (nullable = true)
 |-- usage_lag_1: double (nullable = true)
 |-- usage_lag_7: double (nullable = true)
 |-- week_over_week_growth: double (nullable = true)
 |-- seasonality_factor: double (nullable = true)
 |-- d_daily_usage_units: integer (nullable = true)
 |-- d_date: date (nullable = true)
 |-- d_downtime_min: integer (nullable = true)
 |-- d_econ_index: integer (nullable = true)
 |-- d_peak_usage_units: integer (nullable = true)
 |-- d_region: string (nullable = true)
 |-- d_season: string (nullable = true)
 |-- d_service: string (nullable = true)
 |-- d_storage_tb: integer (nullab

date,region,service,daily_usage_units,peak_usage_units,vm_count,storage_tb,season,econ_index,downtime_min,usage_lag_1,usage_lag_7,week_over_week_growth,seasonality_factor,d_daily_usage_units,d_date,d_downtime_min,d_econ_index,d_peak_usage_units,d_region,d_season,d_service,d_storage_tb,d_vm_count,external_date,cloud_demand_index,gdp_growth,inflation,competitor_price_index
2022-10-06,East US,Compute,157899,175674,10465,0,Autumn,109,0,79757.0,105464.0,49.71838731699916,1.1379450666416375,157899,2022-10-06,0,109,175674,East US,Autumn,Compute,0,10465,2022-10-06,109,3.9173546029233512,5.578609903773472,117
2023-09-11,East US,Compute,166349,195137,13989,0,Autumn,92,1,85447.0,117923.0,41.065780212511555,1.1743873636069189,166349,2023-09-11,1,92,195137,East US,Autumn,Compute,0,13989,2023-09-11,88,3.9108112837826705,7.406119690793604,92
2020-02-10,East US,Storage,0,0,0,855,Winter,109,0,0.0,0.0,0.0,1.1490928323493095,0,2020-02-10,0,109,0,East US,Winter,Storage,855,0,2020-02-10,73,2.2035604657016576,7.911764036647351,92


In [0]:
from pyspark.sql.functions import col

gold_cleaned = (
    gold_step2
    .select(
        # Main dimensions
        col("date").alias("date"),
        col("region").alias("region"),
        col("service").alias("service"),

        # Feature engineered metrics
        col("daily_usage_units").alias("usage_units"),
        col("peak_usage_units"),
        col("vm_count"),
        col("storage_tb"),
        col("season"),
        col("econ_index"),
        col("downtime_min"),
        col("usage_lag_1"),
        col("usage_lag_7"),
        col("week_over_week_growth"),
        col("seasonality_factor"),

        # Demand actual from Demand Silver
        col("d_daily_usage_units").alias("demand_actual"),

        # External factors
        col("cloud_demand_index"),
        col("gdp_growth"),
        col("inflation"),
        col("competitor_price_index")
    )
)

print("FINAL GOLD ROW COUNT:", gold_cleaned.count())
gold_cleaned.printSchema()
display(gold_cleaned.limit(5))


FINAL GOLD ROW COUNT: 10962
root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- service: string (nullable = true)
 |-- usage_units: integer (nullable = true)
 |-- peak_usage_units: integer (nullable = true)
 |-- vm_count: integer (nullable = true)
 |-- storage_tb: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- econ_index: integer (nullable = true)
 |-- downtime_min: integer (nullable = true)
 |-- usage_lag_1: double (nullable = true)
 |-- usage_lag_7: double (nullable = true)
 |-- week_over_week_growth: double (nullable = true)
 |-- seasonality_factor: double (nullable = true)
 |-- demand_actual: integer (nullable = true)
 |-- cloud_demand_index: integer (nullable = true)
 |-- gdp_growth: double (nullable = true)
 |-- inflation: double (nullable = true)
 |-- competitor_price_index: integer (nullable = true)



date,region,service,usage_units,peak_usage_units,vm_count,storage_tb,season,econ_index,downtime_min,usage_lag_1,usage_lag_7,week_over_week_growth,seasonality_factor,demand_actual,cloud_demand_index,gdp_growth,inflation,competitor_price_index
2022-10-06,East US,Compute,157899,175674,10465,0,Autumn,109,0,79757.0,105464.0,49.71838731699916,1.1379450666416375,157899,109,3.9173546029233512,5.578609903773472,117
2023-09-11,East US,Compute,166349,195137,13989,0,Autumn,92,1,85447.0,117923.0,41.065780212511555,1.1743873636069189,166349,88,3.9108112837826705,7.406119690793604,92
2020-02-10,East US,Storage,0,0,0,855,Winter,109,0,0.0,0.0,0.0,1.1490928323493095,0,73,2.2035604657016576,7.911764036647351,92
2020-05-08,East US,Storage,0,0,0,977,Spring,100,1,0.0,0.0,0.0,1.0769441644619129,0,117,4.61814317588024,7.450322288235068,112
2021-02-02,East US,Storage,0,0,0,743,Winter,104,0,0.0,0.0,0.0,1.054067234234836,0,99,3.963777906046026,7.474451532852226,92


In [0]:
gold_path = "abfss://gold@azuresupply2605.dfs.core.windows.net/final_gold/"

(
    gold_cleaned
    .write
    .mode("overwrite")
    .format("delta")
    .save(gold_path)
)

print("Gold table saved to:", gold_path)


Gold table saved to: abfss://gold@azuresupply2605.dfs.core.windows.net/final_gold/


In [0]:
display(gold_cleaned)

date,region,service,usage_units,peak_usage_units,vm_count,storage_tb,season,econ_index,downtime_min,usage_lag_1,usage_lag_7,week_over_week_growth,seasonality_factor,demand_actual,cloud_demand_index,gdp_growth,inflation,competitor_price_index
2022-10-06,East US,Compute,157899,175674,10465,0,Autumn,109,0,79757.0,105464.0,49.71838731699916,1.1379450666416375,157899,109,3.9173546029233512,5.578609903773472,117
2023-09-11,East US,Compute,166349,195137,13989,0,Autumn,92,1,85447.0,117923.0,41.065780212511555,1.1743873636069189,166349,88,3.9108112837826705,7.406119690793604,92
2020-02-10,East US,Storage,0,0,0,855,Winter,109,0,0.0,0.0,0.0,1.1490928323493095,0,73,2.2035604657016576,7.911764036647351,92
2020-05-08,East US,Storage,0,0,0,977,Spring,100,1,0.0,0.0,0.0,1.0769441644619129,0,117,4.61814317588024,7.450322288235068,112
2021-02-02,East US,Storage,0,0,0,743,Winter,104,0,0.0,0.0,0.0,1.054067234234836,0,99,3.963777906046026,7.474451532852226,92
2022-03-26,East US,Storage,0,0,0,1102,Spring,83,0,0.0,0.0,0.0,1.1879911105289334,0,91,5.399393641013738,5.01389843237955,122
2022-03-29,East US,Storage,0,0,0,341,Spring,100,3,0.0,0.0,0.0,1.1365041141134038,0,85,5.863964660370378,3.90127496453396,111
2024-01-15,East US,Storage,0,0,0,981,Winter,81,2,0.0,0.0,0.0,1.0952432973536337,0,97,2.440428676452022,7.271043882149473,103
2024-05-19,East US,Storage,0,0,0,1043,Spring,108,3,0.0,0.0,0.0,1.0668202504734532,0,90,4.700954395097778,6.92730702601401,95
2020-04-01,West Europe,Compute,52315,63054,10191,0,Spring,89,0,97539.0,69654.0,-24.89304275418497,1.0668851732997189,52315,60,3.005853345846478,3.053853268446478,120


In [0]:
%pip install pmdarima prophet xgboost joblib scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
